In [8]:
!pip install python-louvain community -q

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations
import community as community_louvain
from networkx.algorithms.community import girvan_newman
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


def load_data():
    """JSON adatok betöltése"""
    print("Adatok betöltése...")
    try:
        with open("/content/drive/MyDrive/Saját dokumentumok/MSc/Közösségi és Technológiai hálózatok/train.json", "r") as f:
            train_data = json.load(f)
        with open("/content/drive/MyDrive/Saját dokumentumok/MSc/Közösségi és Technológiai hálózatok/test.json", "r") as f:
            test_data = json.load(f)
        return train_data, test_data
    except FileNotFoundError:
        print("HIBA: A fájlok nem találhatók az elérési úton. Ellenőrizd a path-t!")
        return [], []

train_data, test_data = load_data()

def prepare_data(data):
    """DataFrame létrehozása"""
    df = pd.DataFrame(data)
    return df

def create_ingredient_network(df, min_cooccurrence=5):
    """Hozzávalók közötti kapcsolatok gráfjának építése"""
    G = nx.Graph()

    ingredient_counter = Counter()
    ingredient_pairs = Counter()

    for ingredients in df['ingredients']:
        normalized_ingredients = [ing.lower().strip() for ing in ingredients]

        for ingredient in normalized_ingredients:
            ingredient_counter[ingredient] += 1
            G.add_node(ingredient, count=ingredient_counter[ingredient])

        for ing1, ing2 in combinations(sorted(normalized_ingredients), 2):
            ingredient_pairs[(ing1, ing2)] += 1

    for (ing1, ing2), weight in ingredient_pairs.items():
        if weight >= min_cooccurrence:
            G.add_edge(ing1, ing2, weight=weight)

    return G

def create_cuisine_networks(df, min_cooccurrence=3):
    """Konyhatípus specifikus gráfok építése"""
    cuisine_graphs = {}
    if 'cuisine' in df.columns:
        for cuisine in df['cuisine'].unique():
            cuisine_df = df[df['cuisine'] == cuisine]
            cuisine_graphs[cuisine] = create_ingredient_network(cuisine_df, min_cooccurrence)
    return cuisine_graphs

def calculate_centralities(graph):
    """Minden szükséges központisági mérték kiszámítása"""
    degree_centrality = nx.degree_centrality(graph)
    closeness_centrality = nx.closeness_centrality(graph)
    betweenness_centrality = nx.betweenness_centrality(graph)
    clustering_coefficient = nx.clustering(graph)

    # PageRank
    pagerank = nx.pagerank(graph, alpha=0.85)

    # HITS
    try:
        hubs, authorities = nx.hits(graph, max_iter=1000)
    except:
        hubs = {node: 0 for node in graph.nodes()}
        authorities = {node: 0 for node in graph.nodes()}

    # Eigenvector
    try:
        eigenvector_centrality = nx.eigenvector_centrality(graph, max_iter=1000)
    except:
        eigenvector_centrality = {node: 0 for node in graph.nodes()}

    return {
        'degree': degree_centrality,
        'closeness': closeness_centrality,
        'betweenness': betweenness_centrality,
        'eigenvector': eigenvector_centrality,
        'clustering': clustering_coefficient,
        'pagerank': pagerank,
        'hubs': hubs,
        'authorities': authorities,
        'top_degree': sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:20],
        'top_betweenness': sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:20],
        'top_pagerank': sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:20],
        'top_hubs': sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:20],
        'top_authorities': sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:20]
    }

def detect_communities(graph):
    """Közösségdetektálás"""
    # Louvain
    try:
        partition = community_louvain.best_partition(graph)
        modularity = community_louvain.modularity(partition, graph)
    except:
        partition = {node: 0 for node in graph.nodes()}
        modularity = 0

    # Girvan-Newman
    try:
        communities_gn = girvan_newman(graph)
        top_level_communities = next(communities_gn)
        community_list_gn = [list(community) for community in top_level_communities]
    except:
        community_list_gn = [list(graph.nodes())]

    return {
        'louvain_partition': partition,
        'girvan_newman_communities': community_list_gn,
        'modularity': modularity,
        'num_communities': len(set(partition.values()))
    }

def calculate_graph_metrics(graph):
    """Gráf metrikák"""
    metrics = {
        'num_nodes': len(graph.nodes()),
        'num_edges': len(graph.edges()),
        'density': nx.density(graph),
        'avg_degree': sum(dict(graph.degree()).values()) / len(graph.nodes()) if len(graph.nodes()) > 0 else 0,
        'avg_clustering': nx.average_clustering(graph),
        'transitivity': nx.transitivity(graph),
    }

    # Largest connected component metrics
    if len(graph.nodes()) > 0:
        largest_cc = max(nx.connected_components(graph), key=len)
        subgraph = graph.subgraph(largest_cc)
        metrics['diameter_largest_component'] = nx.diameter(subgraph) if len(largest_cc) > 1 else 0
        metrics['avg_path_length_largest_component'] = nx.average_shortest_path_length(subgraph) if len(largest_cc) > 1 else 0
    else:
        metrics['diameter_largest_component'] = 0
        metrics['avg_path_length_largest_component'] = 0

    return metrics

def simulate_influence_spread(graph, seed_nodes, steps=5, propagation_prob=0.3):
    """Independent Cascade Model"""
    influenced = set(seed_nodes)
    newly_influenced = set(seed_nodes)
    spread_over_time = [len(influenced)]

    for step in range(steps):
        next_influenced = set()
        for node in newly_influenced:
            for neighbor in graph.neighbors(node):
                if neighbor not in influenced:
                    if np.random.random() < propagation_prob:
                        next_influenced.add(neighbor)

        influenced.update(next_influenced)
        newly_influenced = next_influenced
        spread_over_time.append(len(influenced))

        if len(next_influenced) == 0:
            break

    return influenced, spread_over_time

def analyze_influence_spread(graph, centralities, num_simulations=50):
    """Hatásterjedés elemzése"""
    top_degree_nodes = [node for node, _ in centralities['top_degree'][:5]]
    top_betweenness_nodes = [node for node, _ in centralities['top_betweenness'][:5]]
    top_pagerank_nodes = [node for node, _ in centralities['top_pagerank'][:5]]
    random_nodes = list(np.random.choice(list(graph.nodes()), 5, replace=False))

    def simulate_multiple_times(seed_node, num_sims=num_simulations):
        all_spreads = []
        for _ in range(num_sims):
            _, spread = simulate_influence_spread(graph, [seed_node], steps=10, propagation_prob=0.25)
            all_spreads.append(spread)
        max_len = max(len(s) for s in all_spreads)
        avg_spread = []
        for i in range(max_len):
            step_values = [s[i] if i < len(s) else s[-1] for s in all_spreads]
            avg_spread.append(np.mean(step_values))
        return avg_spread

    results = {
        'Top Degree': {}, 'Top Betweenness': {}, 'Top PageRank': {}, 'Random': {}
    }
    for node in top_degree_nodes[:3]: results['Top Degree'][node] = simulate_multiple_times(node)
    for node in top_betweenness_nodes[:3]: results['Top Betweenness'][node] = simulate_multiple_times(node)
    for node in top_pagerank_nodes[:3]: results['Top PageRank'][node] = simulate_multiple_times(node)
    for node in random_nodes[:3]: results['Random'][node] = simulate_multiple_times(node)
    return results

def print_initial_statistics(train_df, G, metrics):
    """Statisztikák kiírása"""
    print("\n" + "="*80)
    print("ALAPADATOK:")
    print("="*80)

    # Számoljuk meg az összes hozzávalót
    all_ingredients = []
    for ingredients in train_df['ingredients']:
        all_ingredients.extend([ing.lower().strip() for ing in ingredients])

    print(f"Train adatok: {len(train_df)} recept")
    print(f"Különböző konyhák: {len(train_df['cuisine'].unique())}")
    print(f"Összes hozzávaló előfordulás: {len(all_ingredients)}")

    print(f"Csomópontok száma: {metrics['num_nodes']}")
    print(f"Élek száma: {metrics['num_edges']}")
    print(f"Sűrűség: {metrics['density']:.4f}")
    print(f"Átlagos fokszám: {metrics['avg_degree']:.4f}")
    print(f"Átlagos klaszterezési együttható: {metrics['avg_clustering']:.4f}")
    print(f"Transzitivitás: {metrics['transitivity']:.4f}")
    print(f"Legnagyobb komponens átmérője: {metrics['diameter_largest_component']}")
    print(f"Legnagyobb komponens átlagos úthossz: {metrics['avg_path_length_largest_component']:.4f}")


# 1. PONT: Hozzávalók Hálózata
def visualize_ingredient_network(graph, centralities, communities):
    """1. Hozzávalók Hálózata"""

    important_nodes = set()
    for node, _ in centralities['top_betweenness'][:50]:
        important_nodes.add(node)
    for node in list(important_nodes):
        for neighbor in graph.neighbors(node):
            if len(important_nodes) < 200:
                important_nodes.add(neighbor)

    subgraph = graph.subgraph(important_nodes)
    if len(subgraph.nodes()) > 0:
        pos = nx.spring_layout(subgraph, k=1, iterations=50, seed=42)
        edge_trace = []
        for edge in subgraph.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_weight = subgraph[edge[0]][edge[1]]['weight']
            edge_trace.append(go.Scatter(
                x=[x0, x1, None], y=[y0, y1, None], mode='lines',
                line=dict(width=0.5, color='#888'), hoverinfo='text',
                text=f'{edge[0]} ↔ {edge[1]}<br>Súly: {edge_weight}', showlegend=False
            ))

        node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
        for node in subgraph.nodes():
            x, y = pos[node]
            node_x.append(x); node_y.append(y)
            community_id = communities['louvain_partition'].get(node, 0)
            degree = centralities['degree'].get(node, 0)
            node_text.append(f"<b>{node}</b><br>Közösség: {community_id}<br>Degree: {degree:.4f}")
            node_color.append(community_id)
            node_size.append(max(10, degree * 50))

        node_trace = go.Scatter(
            x=node_x, y=node_y, mode='markers+text', hoverinfo='text',
            text=[node if centralities['betweenness'].get(node, 0) > 0.01 else '' for node in subgraph.nodes()],
            textposition="top center", textfont=dict(size=8), hovertext=node_text,
            marker=dict(showscale=False, colorscale='Viridis', color=node_color, size=node_size, line=dict(width=2, color='white'))
        )

        fig = go.Figure(data=edge_trace + [node_trace],
            layout=go.Layout(title='<b>1. Hozzávalók Hálózata</b>',
            showlegend=False, hovermode='closest', height=700, plot_bgcolor='rgba(240,240,240,0.9)',
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
        fig.show()

# 2. PONT: Top 15 Legfontosabb Hozzávaló
def visualize_top_ingredients(centralities):
    """2. Top 15 Legfontosabb Hozzávaló"""

    top_ingredients = [ing[0] for ing in centralities['top_degree'][:15]]
    top_degrees = [ing[1] for ing in centralities['top_degree'][:15]]
    top_betweenness = [centralities['betweenness'][ing] for ing in top_ingredients]

    fig = go.Figure()
    fig.add_trace(go.Bar(y=top_ingredients, x=top_degrees, name='Degree Centrality', orientation='h', marker=dict(color='lightcoral')))
    fig.add_trace(go.Bar(y=top_ingredients, x=top_betweenness, name='Betweenness Centrality', orientation='h', marker=dict(color='lightskyblue')))
    fig.update_layout(title='<b>2. Top 15 Legfontosabb Hozzávaló</b>',
        xaxis_title='Centrality Érték', yaxis_title='Hozzávaló', barmode='group', height=600)
    fig.show()

# 3. PONT: Top 20 Legerősebb Kapcsolat
def visualize_top_connections(graph):
    """3. Top 20 Legerősebb Kapcsolat"""

    edge_weights = [(graph[u][v]['weight'], u, v) for u, v in graph.edges()]
    edge_weights.sort(reverse=True)
    top_edges = edge_weights[:20]

    if top_edges:
        edge_df = pd.DataFrame(top_edges, columns=['Súly', 'Hozzávaló 1', 'Hozzávaló 2'])
        edge_df['Kapcsolat'] = edge_df['Hozzávaló 1'] + ' ↔ ' + edge_df['Hozzávaló 2']
        fig = go.Figure(go.Bar(
            y=edge_df['Kapcsolat'], x=edge_df['Súly'], orientation='h',
            marker=dict(color=edge_df['Súly'], colorscale='Reds', showscale=True, colorbar=dict(title="Súly"))
        ))
        fig.update_layout(title='<b>3. Top 20 Legerősebb Kapcsolat</b>', height=700, yaxis=dict(autorange="reversed"))
        fig.show()

# 4. PONT: Konyhák Hálózati Jellemzői
def visualize_cuisine_characteristics(cuisine_graphs):
    """4. Konyhák Hálózati Jellemzői"""

    cuisine_data = []
    for cuisine, g in cuisine_graphs.items():
        if len(g.nodes()) > 0:
            cuisine_data.append({
                'Konyha': cuisine, 'Hozzávalók száma': len(g.nodes()),
                'Kapcsolatok száma': len(g.edges()), 'Sűrűség': nx.density(g)
            })
    cuisine_df = pd.DataFrame(cuisine_data).sort_values('Sűrűség', ascending=True)

    fig = make_subplots(rows=1, cols=2, subplot_titles=('Hozzávalók és Kapcsolatok', 'Hálózat Sűrűség'), specs=[[{'type': 'bar'}, {'type': 'bar'}]])
    fig.add_trace(go.Bar(y=cuisine_df['Konyha'], x=cuisine_df['Hozzávalók száma'], name='Hozzávalók', orientation='h', marker=dict(color='lightgreen')), row=1, col=1)
    fig.add_trace(go.Bar(y=cuisine_df['Konyha'], x=cuisine_df['Kapcsolatok száma'], name='Kapcsolatok', orientation='h', marker=dict(color='lightblue')), row=1, col=1)
    fig.add_trace(go.Bar(y=cuisine_df['Konyha'], x=cuisine_df['Sűrűség'], name='Sűrűség', orientation='h', marker=dict(color='salmon'), showlegend=False), row=1, col=2)
    fig.update_layout(title='<b>4. Konyhák Hálózati Jellemzői</b>', height=700)
    fig.show()

# 5. PONT: Hálózat Robosztussága
def visualize_network_robustness(graph, centralities):
    """5. Hálózat Robosztussága"""

    initial_largest = len(max(nx.connected_components(graph), key=len))
    top_nodes = [node for node, _ in centralities['top_betweenness'][:3]]
    removal_data = []

    for node in top_nodes:
        if node in graph:
            G_temp = graph.copy()
            G_temp.remove_node(node)
            curr_largest = len(max(nx.connected_components(G_temp), key=len))
            fragmentation = 1 - (curr_largest / initial_largest)
            removal_data.append({'Node': node, 'Fragmentation': fragmentation, 'Components': nx.number_connected_components(G_temp)})

    df_rem = pd.DataFrame(removal_data)
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Fragmentation', 'Komponensek Száma'))
    fig.add_trace(go.Bar(x=df_rem['Node'], y=df_rem['Fragmentation'], marker_color='crimson'), row=1, col=1)
    fig.add_trace(go.Bar(x=df_rem['Node'], y=df_rem['Components'], marker_color='orange'), row=1, col=2)
    fig.update_layout(title='<b>5. Hálózat Robusztussága</b>', height=500, showlegend=False)
    fig.show()

# 6. PONT: Girvan-Newman Közösségek Struktúrája
def visualize_gn_communities(graph, centralities, communities):
    """6. Girvan-Newman Közösségek Struktúrája"""

    gn_communities = communities['girvan_newman_communities']
    valid_communities = [comm for comm in gn_communities if len(comm) > 0]
    treemap_data = []

    for i, community in enumerate(valid_communities[:15]):
        sorted_comm = sorted(list(community), key=lambda x: centralities['pagerank'].get(x, 0), reverse=True)
        for node in sorted_comm[:25]:
            treemap_data.append({
                'Közösség': f'Csoport {i+1} ({len(community)} elem)',
                'Hozzávaló': node,
                'PageRank': centralities['pagerank'].get(node, 0),
                'Degree': centralities['degree'].get(node, 0),
                'Méret': max(0.001, centralities['pagerank'].get(node, 0))
            })

    if treemap_data:
        df_tree = pd.DataFrame(treemap_data)
        fig = px.treemap(
            df_tree, path=['Közösség', 'Hozzávaló'], values='Méret', color='Degree',
            color_continuous_scale='RdYlGn',
            title='<b>6. Girvan-Newman Közösségek Struktúrája</b>',
            hover_data=['PageRank', 'Degree']
        )
        fig.update_layout(height=700)
        fig.show()

# 7. PONT: Központisági Mértékek Összehasonlítása (Normalizált)
def visualize_centrality_comparison(centralities):
    """7. Központisági Mértékek Összehasonlítása (Normalizált)"""

    top_ingredients = list(set([ing[0] for ing in centralities['top_degree'][:20]] +
                               [ing[0] for ing in centralities['top_betweenness'][:20]] +
                               [ing[0] for ing in centralities['top_pagerank'][:20]]))

    data = []
    for ing in top_ingredients:
        data.append({'Ingredient': ing, 'Degree': centralities['degree'][ing],
                     'Betweenness': centralities['betweenness'][ing], 'Closeness': centralities['closeness'][ing],
                     'PageRank': centralities['pagerank'][ing]})

    df = pd.DataFrame(data).set_index('Ingredient')
    df_norm = (df - df.min()) / (df.max() - df.min())

    fig = px.imshow(df_norm, x=['Degree', 'Betweenness', 'Closeness', 'PageRank'], y=df_norm.index,
                    color_continuous_scale='Viridis', aspect="auto",
                    title="<b>7. Központisági Mértékek Összehasonlítása (Normalizált)</b>")
    fig.update_layout(height=800)
    fig.show()

# 8. PONT: Hatásterjedés Szimuláció
def visualize_influence_spread(graph, centralities):
    """8. Hatásterjedés Szimuláció"""

    results = analyze_influence_spread(graph, centralities)

    fig = go.Figure()
    colors = {'Top Degree': 'red', 'Top Betweenness': 'blue', 'Top PageRank': 'green', 'Random': 'gray'}

    for category, nodes_data in results.items():
        for node, spread in nodes_data.items():
            fig.add_trace(go.Scatter(x=list(range(len(spread))), y=spread, mode='lines+markers',
                                     name=f'{category}: {node}', line=dict(color=colors[category]), marker=dict(size=8)))

    fig.update_layout(title='<b>8. Hatásterjedés Szimuláció</b><br>',
                      xaxis_title='Terjedési lépés', yaxis_title='Befolyásolt csúcsok', height=700)
    fig.show()

# 9. PONT: HITS Algoritmus: Hubs vs Authorities
def visualize_hits_algorithm(centralities):
    """9. HITS Algoritmus: Hubs vs Authorities"""

    top_ingredients = list(set([ing[0] for ing in centralities['top_hubs'][:15]] + [ing[0] for ing in centralities['top_authorities'][:15]]))[:20]
    hub_vals = [centralities['hubs'][ing] for ing in top_ingredients]
    auth_vals = [centralities['authorities'][ing] for ing in top_ingredients]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hub_vals, y=auth_vals, mode='markers+text', text=top_ingredients, textposition='top center',
                             marker=dict(size=[max(10, (h+a)*200) for h, a in zip(hub_vals, auth_vals)],
                                         color=hub_vals, colorscale='Turbo', showscale=True, colorbar=dict(title="Hub Score"))))
    fig.update_layout(title='<b>9. HITS Algoritmus: Hubs vs Authorities</b><br>',
                      xaxis_title='Hub Score', yaxis_title='Authority Score', height=700)
    fig.show()

if __name__ == "__main__":
    # Adatok betöltése
    train_data, _ = load_data()
    train_df = prepare_data(train_data)

    if not train_df.empty:
        print(f"Betöltve: {len(train_df)} recept.")
        print(f"Különböző konyhák: {len(train_df['cuisine'].unique())}")

        # Gráfok építése
        print("\nGráfok építése...")
        G = create_ingredient_network(train_df, min_cooccurrence=10)
        cuisine_graphs = create_cuisine_networks(train_df, min_cooccurrence=3)

        # Gráf metrikák számítása
        print("Gráf metrikák számítása...")
        metrics = calculate_graph_metrics(G)

        # Statisztikák kiírása
        print_initial_statistics(train_df, G, metrics)

        # Központisági mértékek számítása
        print("Központisági mértékek számítása...")
        centralities = calculate_centralities(G)

        # Közösségdetektálás
        print("Közösségdetektálás...")
        communities = detect_communities(G)

        # 1. Hozzávalók Hálózata
        visualize_ingredient_network(G, centralities, communities)

        # 2. Top 15 Legfontosabb Hozzávaló
        visualize_top_ingredients(centralities)

        # 3. Top 20 Legerősebb Kapcsolat
        visualize_top_connections(G)

        # 4. Konyhák Hálózati Jellemzői
        visualize_cuisine_characteristics(cuisine_graphs)

        # 5. Hálózat Robosztussága
        visualize_network_robustness(G, centralities)

        # 6. Girvan-Newman Közösségek Struktúrája
        visualize_gn_communities(G, centralities, communities)

        # 7. Központisági Mértékek Összehasonlítása (Normalizált)
        visualize_centrality_comparison(centralities)

        # 8. Hatásterjedés Szimuláció
        visualize_influence_spread(G, centralities)

        # 9. HITS Algoritmus: Hubs vs Authorities
        visualize_hits_algorithm(centralities)

Adatok betöltése...
Adatok betöltése...
Betöltve: 39774 recept.
Különböző konyhák: 20

Gráfok építése...
Gráf metrikák számítása...

ALAPADATOK:
Train adatok: 39774 recept
Különböző konyhák: 20
Összes hozzávaló előfordulás: 428275
Csomópontok száma: 6703
Élek száma: 39679
Sűrűség: 0.0018
Átlagos fokszám: 11.8392
Átlagos klaszterezési együttható: 0.2022
Transzitivitás: 0.2554
Legnagyobb komponens átmérője: 4
Legnagyobb komponens átlagos úthossz: 2.0739
Központisági mértékek számítása...
Közösségdetektálás...
